In [2]:
import numpy as np 
import pandas as pd 
import torch 
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder
import torch.nn as nn

In [3]:
df = pd.read_csv('https://raw.githubusercontent.com/gscdit/Breast-Cancer-Detection/refs/heads/master/data.csv')
df.head()

,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst,Unnamed: 32
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,NaN
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,NaN
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,NaN
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,NaN
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,NaN


In [4]:
df.drop(columns=['id','Unnamed: 32'] , inplace=True)

In [5]:
X_train,X_test,Y_train,Y_test = train_test_split(df.iloc[:,1:],df.iloc[:,0],test_size=0.2)

In [6]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [7]:
encoder = LabelEncoder()
y_train = encoder.fit_transform(Y_train)

In [8]:

y_test = encoder.transform(Y_test)


In [9]:
# conv to tensor vectors
X_train_tensor = torch.from_numpy(X_train).float()
X_test_tensor = torch.from_numpy(X_test).float()
y_train_tensor = torch.from_numpy(y_train).float()
y_test_tensor = torch.from_numpy(y_test).float()

In [14]:
from torch.utils.data import Dataset,DataLoader

class CustomDatasetClass(Dataset):
    def __init__(self,features,labels):
        self.features = features
        self.labels = labels
    def __len__(self):
        return len(self.features)
    def __getitem__(self, index):
        return self.features[index],self.labels[index]

In [15]:
train_dataset = CustomDatasetClass(X_train_tensor,y_train_tensor)
test_dataset = CustomDatasetClass(X_test_tensor,y_test_tensor)

In [17]:
train_loader = DataLoader(train_dataset , batch_size = 32 , shuffle= True)
test_loader = DataLoader(test_dataset , batch_size = 32 , shuffle= True)


In [10]:
learning_rate = 0.1
epochs = 25 

In [11]:
class MySimpleNN2(nn.Module):
    def __init__(self,num_features):
        super().__init__()
        self.linear = nn.Linear(num_features,1)
        self.sigmoid = nn.Sigmoid()
    def forward(self,features):
        out = self.linear(features)
        out = self.sigmoid(out)
        return out
    

In [20]:
model = MySimpleNN2(X_train_tensor.shape[1])
loss_function = nn.BCELoss()
optimzer = torch.optim.SGD(model.parameters() , lr = learning_rate)

# forward pass
for epoch in range(epochs):
    for batch_features, batch_lables in train_loader:
        # forward pass
        y_pred = model(batch_features)
        #loss
        loss = loss_function(y_pred , batch_lables.view(-1,1))
        #clear grads
        optimzer.zero_grad()
        #backward pass
        loss.backward
        #parameters update
        optimzer.step()
        ##print loss in each epoch
        print(f"Epoch : {epoch+1} Loss : {loss.item()}")
    

Epoch : 1 Loss : 0.6100282073020935
Epoch : 1 Loss : 0.6011741757392883
Epoch : 1 Loss : 0.5769952535629272
Epoch : 1 Loss : 0.5653252005577087
Epoch : 1 Loss : 0.5251325368881226
Epoch : 1 Loss : 0.5580544471740723
Epoch : 1 Loss : 0.5901408195495605
Epoch : 1 Loss : 0.5662533044815063
Epoch : 1 Loss : 0.5701126456260681
Epoch : 1 Loss : 0.5874652862548828
Epoch : 1 Loss : 0.5403496026992798
Epoch : 1 Loss : 0.583489716053009
Epoch : 1 Loss : 0.569490373134613
Epoch : 1 Loss : 0.5607538819313049
Epoch : 1 Loss : 0.5257325768470764
Epoch : 2 Loss : 0.5626893043518066
Epoch : 2 Loss : 0.5536922812461853
Epoch : 2 Loss : 0.5780353546142578
Epoch : 2 Loss : 0.5674225091934204
Epoch : 2 Loss : 0.5607634782791138
Epoch : 2 Loss : 0.563559889793396
Epoch : 2 Loss : 0.5450791120529175
Epoch : 2 Loss : 0.5799891948699951
Epoch : 2 Loss : 0.5683659315109253
Epoch : 2 Loss : 0.6185950636863708
Epoch : 2 Loss : 0.5259419083595276
Epoch : 2 Loss : 0.6133224964141846
Epoch : 2 Loss : 0.628101527690

In [19]:
#model eval using test loader
model.eval()  # set model to eval mode
accuracy_list = []

with torch.no_grad():
    for batch_features,batch_lables in test_loader:
        #forward pass
        y_pred = model(batch_features)
        y_pred = (y_pred > 0.8).float()

        # calc accuracy for current batch
        batch_accuracy = (y_pred.view(-1)==batch_lables).float().mean().item()
        accuracy_list.append(batch_accuracy)

# overall accuracy
overall_acc = sum(accuracy_list)/len(accuracy_list)
print(f"accuracy : {overall_acc: .4f}")

accuracy :  0.5729
